## Inference + Evaluation
Notebook ini menjalankan routed inference menggunakan `master_model.joblib`, lalu menampilkan:
- hasil prediksi dan alasan SHAP yang sudah dirapikan
- statistik distribusi prediksi per label dan per daerah
- consistency accuracy terhadap reference outputs yang sudah tersimpan
- catatan eksplisit bahwa true accuracy hanya bisa dihitung jika ada ground-truth label audit/fraud


In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import shap
from pathlib import Path
from IPython.display import Markdown, display

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


def print_section(title):
    print("=" * 80)
    print(title)
    print("=" * 80)


artifact_dir = Path("../artifacts/post_award_anomaly")
test_data_dir = Path("../data/cleaned/test")
training_summary_path = artifact_dir / "master_training_summary.csv"
reference_output_path = artifact_dir / "master_test_predictions_with_explanations.csv"

print_section("LOADING MASTER BUNDLE, TEST DATA, AND REFERENCE OUTPUTS")
master_bundle = joblib.load(artifact_dir / "master_model.joblib")
training_summary = pd.read_csv(training_summary_path)
reference_output = pd.read_csv(reference_output_path)

test_files = sorted(test_data_dir.glob("*.csv"))
if not test_files:
    raise FileNotFoundError(f"Tidak menemukan file CSV di {test_data_dir}")

all_test_data = pd.concat([pd.read_csv(file_path) for file_path in test_files], ignore_index=True)
all_test_data["demo_case_label"] = [f"Test_Case_{i+1:04d}" for i in range(len(all_test_data))]

print(f"Bundle Type      : {master_bundle['bundle_type']}")
print(f"Routing Column   : {master_bundle['routing_column']}")
print(f"Total Regions    : {master_bundle['region_count']} regions loaded")
print(f"Rows Test Aktif  : {len(all_test_data):,}")
print(f"Reference Rows   : {len(reference_output):,}")

display(training_summary[[
    "nama_daerah", "train_rows", "test_rows", "cutoff_date",
    "medium_cutoff", "anomaly_threshold", "test_anomaly_rate"
]])


LOADING MASTER BUNDLE, TEST DATA, AND REFERENCE OUTPUTS
Bundle Type      : multi_daerah_isolation_forest_router
Routing Column   : nama_daerah
Total Regions    : 27 regions loaded
Rows Test Aktif  : 12,057
Reference Rows   : 12,057


,nama_daerah,train_rows,test_rows,cutoff_date,medium_cutoff,anomaly_threshold,test_anomaly_rate
0,Gorontalo,NaN,NaN,NaN,NaN,NaN,NaN
1,Kalimantan Barat,NaN,NaN,NaN,NaN,NaN,NaN
2,Kalimantan Selatan,NaN,NaN,NaN,NaN,NaN,NaN
3,Kalimantan Tengah,NaN,NaN,NaN,NaN,NaN,NaN
4,Kepulauan Bangka Belitung,NaN,NaN,NaN,NaN,NaN,NaN
5,Kepulauan Riau,NaN,NaN,NaN,NaN,NaN,NaN
6,Sumatera Barat,NaN,NaN,NaN,NaN,NaN,NaN
7,Aceh,"10,604.0000","1,871.0000",2022-03-21,0.4607,0.5754,0.0096
8,Bali,"1,643.0000",291.0000,2021-06-28,0.4734,0.5883,0.0687
9,Banten,"2,934.0000",518.0000,2022-03-17,0.4637,0.5769,0.0174


In [2]:
# Helper Functions

def format_number(value) -> str:
    if pd.isna(value):
        return "missing"
    if isinstance(value, (int, np.integer)):
        return f"{int(value):,}"
    if isinstance(value, (float, np.floating)):
        return f"{value:,.4f}".rstrip("0").rstrip(".")
    return str(value)


def assign_severity(scores, medium_cutoff: float, anomaly_threshold: float):
    return np.select(
        [scores >= anomaly_threshold, scores >= medium_cutoff],
        ["high", "medium"],
        default="low",
    )


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    data = df.copy()
    if "date" in data.columns:
        data["date"] = pd.to_datetime(data["date"], errors="coerce")
    data["award_date"] = pd.to_datetime(data["award_date"], errors="coerce")

    data["award_month"] = data["award_date"].dt.month
    data["award_quarter"] = data["award_date"].dt.quarter
    data["award_weekday"] = data["award_date"].dt.weekday

    data["log_tender_minvalue"] = np.log1p(pd.to_numeric(data["tender_minvalue"], errors="coerce").clip(lower=0))
    data["log_award_value"] = np.log1p(pd.to_numeric(data["award_value"], errors="coerce").clip(lower=0))
    data["value_gap"] = pd.to_numeric(data["award_value"], errors="coerce") - pd.to_numeric(data["tender_minvalue"], errors="coerce")

    budget_ratio = (
        pd.to_numeric(data["award_value"], errors="coerce") /
        pd.to_numeric(data["tender_minvalue"], errors="coerce").replace(0, np.nan)
    )
    if "budget_utilization_ratio" not in data.columns:
        data["budget_utilization_ratio"] = budget_ratio.fillna(0)
    else:
        data["budget_utilization_ratio"] = pd.to_numeric(data["budget_utilization_ratio"], errors="coerce").fillna(budget_ratio).fillna(0)

    data["title_word_count"] = data["tender_title"].fillna("").astype(str).str.split().str.len()
    data["award_title_word_count"] = data["award_title"].fillna("").astype(str).str.split().str.len()

    tokens = data["award_supplier"].fillna("").astype(str).str.split(",")
    data["supplier_count"] = tokens.apply(lambda items: max(len([item for item in items if str(item).strip()]), 1))

    safe_days = pd.to_numeric(data["days_to_award"], errors="coerce").replace(0, 1)
    data["award_value_per_day"] = pd.to_numeric(data["award_value"], errors="coerce") / safe_days
    data["same_day_award_flag"] = (pd.to_numeric(data["days_to_award"], errors="coerce") == 0).astype(int)
    return data


def normalize_shap_values(shap_values):
    if isinstance(shap_values, list):
        shap_values = shap_values[0]
    return np.asarray(shap_values)


def standardize_merge_keys(frame: pd.DataFrame, key_columns: list[str]) -> pd.DataFrame:
    normalized = frame.copy()
    for column in key_columns:
        merge_column = f"_merge_{column}"
        if column in {"tender_minvalue", "award_value"}:
            normalized[merge_column] = pd.to_numeric(normalized[column], errors="coerce").round(6)
        elif column == "days_to_award":
            normalized[merge_column] = pd.to_numeric(normalized[column], errors="coerce").astype("Int64")
        elif "date" in column:
            normalized[merge_column] = pd.to_datetime(normalized[column], errors="coerce").dt.strftime("%Y-%m-%d")
        else:
            normalized[merge_column] = normalized[column].fillna("").astype(str).str.strip().str.casefold()
    return normalized


KAMUS_KONSEP = {
    "award_title_word_count": "Kompleksitas Judul Kontrak",
    "days_to_award": "Durasi Proses Tender",
    "budget_utilization_ratio": "Rasio Penyerapan Anggaran",
    "value_gap": "Selisih Nilai Tender dan Kontrak",
    "supplier_count": "Jumlah Peserta Tender",
    "award_value_per_day": "Laju Pengeluaran Harian",
    "tender_minvalue": "Batas Minimum Tender",
    "award_value": "Nilai Kontrak Akhir",
    "mainprocurementcategory": "Kategori Pengadaan Utama",
}

KAMUS_RISIKO = {
    "award_title_word_count": "pola penamaan judul yang tidak standar seringkali digunakan untuk mengaburkan spesifikasi asli proyek.",
    "days_to_award": "proses yang selesai terlalu cepat atau terlalu lambat mengindikasikan adanya potensi pengaturan pemenang.",
    "budget_utilization_ratio": "penyerapan yang mendekati 100% secara sempurna merupakan indikator umum terjadinya mark-up harga.",
    "value_gap": "selisih yang tidak proporsional menunjukkan potensi inefisiensi atau kesalahan estimasi biaya awal.",
    "supplier_count": "minimnya partisipan tender dapat mengurangi kompetisi sehat dan meningkatkan risiko monopoli.",
    "award_value_per_day": "beban biaya harian yang ekstrem menunjukkan ketidakwajaran antara nilai proyek dengan durasi pengerjaan.",
    "tender_minvalue": "penetapan batas minimum yang tidak lazim berisiko membatasi partisipasi vendor kompeten.",
    "mainprocurementcategory": "anomali seringkali terpusat pada kategori spesifik ini akibat celah kontrol internal yang kurang pengawasan.",
}


def generate_natural_reason(feat, raw_val, shap_val, severity_band):
    is_anomaly_driver = shap_val > 0
    nama_manusiawi = KAMUS_KONSEP.get(feat, feat.replace("_", " ").title())
    val_str = format_number(raw_val)

    if severity_band == "high":
        if is_anomaly_driver:
            risiko = KAMUS_RISIKO.get(feat, "hal ini memerlukan verifikasi kepatuhan dokumen lebih lanjut.")
            return f"Sistem menemukan indikasi penyimpangan serius pada **{nama_manusiawi}** ({val_str}). Secara audit, {risiko}"
        return f"Meskipun terdapat temuan risiko lain, parameter **{nama_manusiawi}** ({val_str}) terpantau tetap stabil."

    if severity_band == "medium":
        if is_anomaly_driver:
            return f"Terdapat sedikit kejanggalan pada data **{nama_manusiawi}** ({val_str}). Polanya tidak biasa, tetapi masih berada dalam batas toleransi kebijakan."
        return f"Parameter **{nama_manusiawi}** ({val_str}) memberikan sinyal kestabilan di tengah beberapa anomali minor lainnya."

    if is_anomaly_driver:
        return f"Komponen **{nama_manusiawi}** ({val_str}) menunjukkan aktivitas yang dinamis namun tetap sesuai dengan standar operasional."
    return f"Parameter **{nama_manusiawi}** ({val_str}) sangat identik dengan profil pengadaan yang bersih dan akuntabel."


def explain_prediction_shap(row_shap, original_row, feature_names):
    severity_band = original_row["severity_band"]
    top_indices = np.argsort(np.abs(row_shap))[-3:][::-1]

    reasons = []
    for idx in top_indices:
        feat_name = feature_names[idx]
        raw_feat_name = "mainprocurementcategory" if feat_name.startswith("cat_") else feat_name
        raw_val = original_row[raw_feat_name] if raw_feat_name in original_row else "N/A"
        reasons.append(generate_natural_reason(raw_feat_name, raw_val, row_shap[idx], severity_band))

    if severity_band == "high":
        header = "Sistem mendeteksi aktivitas yang MENCURIGAKAN dan berisiko tinggi:"
    elif severity_band == "medium":
        header = "Sistem menemukan beberapa temuan BORDERLINE yang memerlukan perhatian moderat:"
    else:
        header = "Status transaksi dinilai AMAN dan memenuhi kriteria kepatuhan standar:"

    return f"[{severity_band.upper()}] {header}\n" + "\n".join([f"• {reason}" for reason in reasons])


In [3]:
print_section("RUNNING ROUTED INFERENCE ON FULL TEST SET")

full_test_features = engineer_features(all_test_data)
routing_col = master_bundle["routing_column"]
results = []
skipped_regions = []

for region_name, group in full_test_features.groupby(routing_col):
    routing_key = str(region_name).strip().casefold()

    if routing_key not in master_bundle["regions"]:
        skipped_regions.append(region_name)
        continue

    region_bundle = master_bundle["regions"][routing_key]
    model = region_bundle["model"]
    preprocessor = region_bundle["preprocessor"]
    explainer_shap = region_bundle["explainer"]
    config = region_bundle["model_config"]
    meta = region_bundle["explanation_meta"]

    feature_columns = config["feature_columns"]
    feature_names_preprocessed = meta["feature_names_preprocessed"]

    X_group = preprocessor.transform(group[feature_columns])
    group_scores = -model.score_samples(X_group)

    baseline_score = config["medium_cutoff"]
    max_risk_score = config["anomaly_threshold"]
    denominator = max(max_risk_score - baseline_score, 1e-9)
    raw_risk_pct = ((group_scores - baseline_score) / denominator) * 100

    shap_values = normalize_shap_values(explainer_shap.shap_values(X_group))
    if shap_values.ndim == 1:
        shap_values = shap_values.reshape(1, -1)

    group_out = group.copy().reset_index(drop=True)
    group_out["anomaly_score"] = group_scores
    group_out["anomaly_risk_pct"] = np.round(raw_risk_pct, 2)
    group_out["anomaly_risk_str"] = group_out["anomaly_risk_pct"].apply(lambda value: f"{value:+.2f}%")
    group_out["prediction_label"] = np.select(
        [group_scores >= config["anomaly_threshold"], group_scores >= config["medium_cutoff"]],
        ["anomaly", "warning"],
        default="normal",
    )
    group_out["severity_band"] = assign_severity(group_scores, config["medium_cutoff"], config["anomaly_threshold"])
    group_out["anomaly_flag"] = (group_scores >= config["anomaly_threshold"]).astype(int)

    explanations = []
    for row_idx in range(len(group_out)):
        explanations.append(
            explain_prediction_shap(
                shap_values[row_idx],
                group_out.iloc[row_idx],
                feature_names_preprocessed,
            )
        )

    group_out["human_readable_explanation"] = explanations
    results.append(group_out)

if not results:
    raise RuntimeError("Tidak ada hasil inference yang berhasil dihitung.")

final_output = pd.concat(results, ignore_index=True)

comparison_keys = [
    "nama_daerah", "award_date", "tender_title", "award_title", "award_supplier",
    "tender_minvalue", "award_value", "days_to_award", "mainprocurementcategory",
]

reference_compare = standardize_merge_keys(reference_output, comparison_keys)
reference_key_columns = [f"_merge_{column}" for column in comparison_keys]
reference_compare = reference_compare.drop_duplicates(subset=reference_key_columns)
reference_compare = reference_compare.rename(columns={
    "prediction_label": "reference_prediction_label",
    "severity_band": "reference_severity_band",
    "anomaly_score": "reference_anomaly_score",
    "anomaly_flag": "reference_anomaly_flag",
    "human_readable_explanation": "reference_explanation",
})

comparison_output = standardize_merge_keys(final_output, comparison_keys)
final_output = comparison_output.merge(
    reference_compare[
        reference_key_columns + [
            "reference_prediction_label", "reference_severity_band",
            "reference_anomaly_score", "reference_anomaly_flag", "reference_explanation"
        ]
    ],
    on=reference_key_columns,
    how="left",
)

has_reference = final_output["reference_prediction_label"].notna()
final_output["reference_label_match"] = has_reference & (final_output["prediction_label"] == final_output["reference_prediction_label"])
final_output["reference_score_match"] = has_reference & np.isclose(
    final_output["anomaly_score"],
    final_output["reference_anomaly_score"],
    atol=1e-9,
)

output_path = artifact_dir / "master_inference_output_final.csv"
final_output.to_csv(output_path, index=False)

print(f"Hasil inference disimpan ke: {output_path}")
if skipped_regions:
    print(f"Warning: region berikut dilewati karena tidak ada modelnya: {skipped_regions}")

sample_display = final_output.sample(n=min(12, len(final_output)), random_state=42)
display(sample_display[[
    "demo_case_label", "nama_daerah", "award_date", "mainprocurementcategory",
    "award_value", "anomaly_score", "anomaly_risk_str", "prediction_label"
]])


RUNNING ROUTED INFERENCE ON FULL TEST SET
Hasil inference disimpan ke: ..\artifacts\post_award_anomaly\master_inference_output_final.csv


,demo_case_label,nama_daerah,award_date,mainprocurementcategory,award_value,anomaly_score,anomaly_risk_str,prediction_label
10489,Test_Case_7534,Sulawesi Selatan,2022-09-16,Services,"924,297,000.0000",0.4404,-18.33%,normal
2340,Test_Case_11718,Banten,2022-06-30,Services,"721,551,891.6400",0.3998,-56.45%,normal
3635,Test_Case_2734,DKI Jakarta,2022-09-27,Works,"464,954,695.9800",0.4148,-39.47%,normal
4228,Test_Case_3327,DKI Jakarta,2023-09-07,Services,"697,857,000.0000",0.4058,-47.01%,normal
7631,Test_Case_8542,Nusa Tenggara Barat,2022-06-10,Works,"230,491,000.0000",0.4167,-39.48%,normal
2884,Test_Case_5292,Bengkulu,2023-07-05,Works,"497,499,400.6000",0.3794,-87.20%,normal
1213,Test_Case_1317,Aceh,2022-10-27,Works,"433,000,026.6200",0.4287,-27.93%,normal
10526,Test_Case_7571,Sulawesi Selatan,2023-03-01,Services,"127,000,000.0000",0.4550,-5.65%,normal
4447,Test_Case_10428,Jambi,2023-07-12,Works,"396,304,166.3400",0.3938,-57.23%,normal
2144,Test_Case_6700,Bali,2023-07-12,Services,"249,084,000.0000",0.4213,-45.34%,normal


In [4]:
print_section("MODEL PREDICTION STATISTICS AND ACCURACY VIEW")

prediction_stats = (
    final_output["prediction_label"]
    .value_counts(dropna=False)
    .rename_axis("prediction_label")
    .reset_index(name="count")
)
prediction_stats["percentage"] = (prediction_stats["count"] / len(final_output) * 100).round(2)

display(Markdown("### Distribusi Prediksi Keseluruhan"))
display(prediction_stats)

region_prediction_stats = (
    final_output
    .groupby(["nama_daerah", "prediction_label"])
    .size()
    .rename("count")
    .reset_index()
)
region_totals = region_prediction_stats.groupby("nama_daerah")["count"].transform("sum")
region_prediction_stats["percentage"] = (region_prediction_stats["count"] / region_totals * 100).round(2)

display(Markdown("### Distribusi Prediksi per Daerah"))
display(region_prediction_stats)

score_summary = (
    final_output
    .groupby("prediction_label")["anomaly_score"]
    .agg(["count", "mean", "median", "min", "max"])
    .round(4)
    .reset_index()
)

display(Markdown("### Ringkasan Skor per Label"))
display(score_summary)

matched_reference = final_output["reference_prediction_label"].notna()
reference_match_count = int(matched_reference.sum())
reference_true_count = int(final_output.loc[matched_reference, "reference_label_match"].sum())
reference_false_count = int((matched_reference & ~final_output["reference_label_match"]).sum())
reference_score_true_count = int(final_output.loc[matched_reference, "reference_score_match"].sum())
reference_label_accuracy_pct = round((reference_true_count / reference_match_count) * 100, 2) if reference_match_count else np.nan
reference_score_accuracy_pct = round((reference_score_true_count / reference_match_count) * 100, 2) if reference_match_count else np.nan

reference_metrics = pd.DataFrame([
    {"metric": "total_rows_scored", "value": int(len(final_output))},
    {"metric": "reference_rows_found", "value": reference_match_count},
    {"metric": "reference_true_matches", "value": reference_true_count},
    {"metric": "reference_false_mismatches", "value": reference_false_count},
    {"metric": "reference_label_accuracy_pct", "value": reference_label_accuracy_pct},
    {"metric": "reference_score_accuracy_pct", "value": reference_score_accuracy_pct},
])

display(Markdown("### Accuracy View terhadap Reference Output"))
display(reference_metrics)

if reference_match_count:
    confusion_reference = pd.crosstab(
        final_output.loc[matched_reference, "reference_prediction_label"],
        final_output.loc[matched_reference, "prediction_label"],
        rownames=["reference_label"],
        colnames=["current_prediction"],
    )
    display(Markdown("### Confusion Matrix terhadap Reference Output"))
    display(confusion_reference)

region_evaluation_summary = training_summary[[
    "nama_daerah", "train_rows", "test_rows", "cutoff_date",
    "train_anomaly_rate", "test_anomaly_rate"
]].copy()
region_evaluation_summary["test_anomaly_rate_pct"] = (region_evaluation_summary["test_anomaly_rate"] * 100).round(2)

display(Markdown("### Evaluasi Holdout yang Tersedia per Daerah"))
display(region_evaluation_summary)

print("Catatan penting:")
print("- Dataset ini tidak memiliki ground-truth label audit/fraud pada cleaned_data.csv.")
print("- Jadi angka accuracy di notebook ini adalah consistency accuracy terhadap reference output yang sudah tersimpan, bukan real-world truth accuracy.")
print("- Untuk true accuracy, kita butuh label manual seperti fraud / non-fraud atau verified anomaly / clean case.")

prediction_stats.to_csv(artifact_dir / "master_inference_prediction_stats.csv", index=False)
region_prediction_stats.to_csv(artifact_dir / "master_inference_region_stats.csv", index=False)
reference_metrics.to_csv(artifact_dir / "master_inference_reference_metrics.csv", index=False)
print("Summary statistik disimpan ke folder artifacts/post_award_anomaly")


MODEL PREDICTION STATISTICS AND ACCURACY VIEW


### Distribusi Prediksi Keseluruhan

,prediction_label,count,percentage
0,normal,7977,66.1600
1,warning,3592,29.7900
2,anomaly,488,4.0500


### Distribusi Prediksi per Daerah

,nama_daerah,prediction_label,count,percentage
0,Aceh,anomaly,18,0.9600
1,Aceh,normal,1349,72.1000
2,Aceh,warning,504,26.9400
3,Bali,anomaly,20,6.8700
4,Bali,normal,173,59.4500
...,...,...,...,...
75,Sumatera Selatan,normal,100,97.0900
76,Sumatera Selatan,warning,3,2.9100
77,Sumatera Utara,anomaly,12,9.7600
78,Sumatera Utara,normal,59,47.9700


...

### Confusion Matrix terhadap Reference Output

current_prediction,anomaly,normal,warning
reference_label,,,
anomaly,488,0,0
normal,0,7977,0
warning,0,0,3592


### Evaluasi Holdout yang Tersedia per Daerah

,nama_daerah,train_rows,test_rows,cutoff_date,train_anomaly_rate,test_anomaly_rate,test_anomaly_rate_pct
0,Gorontalo,NaN,NaN,NaN,NaN,NaN,NaN
1,Kalimantan Barat,NaN,NaN,NaN,NaN,NaN,NaN
2,Kalimantan Selatan,NaN,NaN,NaN,NaN,NaN,NaN
3,Kalimantan Tengah,NaN,NaN,NaN,NaN,NaN,NaN
4,Kepulauan Bangka Belitung,NaN,NaN,NaN,NaN,NaN,NaN
5,Kepulauan Riau,NaN,NaN,NaN,NaN,NaN,NaN
6,Sumatera Barat,NaN,NaN,NaN,NaN,NaN,NaN
7,Aceh,"10,604.0000","1,871.0000",2022-03-21,0.0301,0.0096,0.9600
8,Bali,"1,643.0000",291.0000,2021-06-28,0.0304,0.0687,6.8700
9,Banten,"2,934.0000",518.0000,2022-03-17,0.0300,0.0174,1.7400


Catatan penting:
- Dataset ini tidak memiliki ground-truth label audit/fraud pada cleaned_data.csv.
- Jadi angka accuracy di notebook ini adalah consistency accuracy terhadap reference output yang sudah tersimpan, bukan real-world truth accuracy.
- Untuk true accuracy, kita butuh label manual seperti fraud / non-fraud atau verified anomaly / clean case.
Summary statistik disimpan ke folder artifacts/post_award_anomaly


In [5]:
print_section("TOP HIGH-RISK CASES FOR DEMO REVIEW")

priority_cases = final_output.sort_values(["anomaly_score", "nama_daerah"], ascending=[False, True]).head(10)

for _, row in priority_cases.iterrows():
    print(f"Kasus: {row['demo_case_label']} | Daerah: {row['nama_daerah']}")
    print(f"Status: {row['prediction_label'].upper()} | Severity: {row['severity_band'].upper()} | Score: {row['anomaly_score']:.4f}")
    if pd.notna(row.get("reference_prediction_label", np.nan)):
        print(f"Reference Label Match: {bool(row['reference_label_match'])} | Reference Label: {row['reference_prediction_label']}")
    print(f"Analisis Sistem:\n{row['human_readable_explanation']}")
    print("-" * 80)


TOP HIGH-RISK CASES FOR DEMO REVIEW
Kasus: Test_Case_2828 | Daerah: DKI Jakarta
Status: ANOMALY | Severity: HIGH | Score: 0.7664
Reference Label Match: True | Reference Label: anomaly
Analisis Sistem:
[HIGH] Sistem mendeteksi aktivitas yang MENCURIGAKAN dan berisiko tinggi:
• Meskipun terdapat temuan risiko lain, parameter **Batas Minimum Tender** (907,898,964,458) terpantau tetap stabil.
• Meskipun terdapat temuan risiko lain, parameter **Selisih Nilai Tender dan Kontrak** (-120,730,944,458) terpantau tetap stabil.
• Meskipun terdapat temuan risiko lain, parameter **Nilai Kontrak Akhir** (787,168,020,000) terpantau tetap stabil.
--------------------------------------------------------------------------------
Kasus: Test_Case_2829 | Daerah: DKI Jakarta
Status: ANOMALY | Severity: HIGH | Score: 0.7642
Reference Label Match: True | Reference Label: anomaly
Analisis Sistem:
[HIGH] Sistem mendeteksi aktivitas yang MENCURIGAKAN dan berisiko tinggi:
• Meskipun terdapat temuan risiko lain, pa